# rl-trading-live — Colab Master Notebook
Run cells 1-6 in order to train. Cell 7 = dashboard, 8 = crash recovery, 9 = GPU profiling.
Code lives in GitHub; data + checkpoints live in Google Drive (Colab is ephemeral).

In [ ]:
# CELL 1 — GPU CHECK
import torch
assert torch.cuda.is_available(), 'NO GPU DETECTED — switch to A100 runtime'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} | VRAM: {vram_gb:.1f}GB')
assert vram_gb > 30, f'Expected A100 (>30GB), got {vram_gb:.1f}GB — check runtime'

In [ ]:
# CELL 2 — MOUNT DRIVE
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
assert os.path.exists('/content/drive/MyDrive/RL-Trading-Data/EURUSD_M1_202101131130_202605270000_2020_2026.csv'), \
    'PRIMARY DATA FILE NOT FOUND — check Google Drive path'
print('Drive mounted. Primary data file confirmed.')

In [ ]:
# CELL 3 — INSTALL DEPENDENCIES (once per session)
# TA-Lib C library must come first (the Python wrapper links against it)
import subprocess, sys
subprocess.run(['apt-get', 'install', '-y', '-q', 'ta-lib'], check=False)
# Install from requirements.txt — pins numpy<2.0 for faiss-cpu compatibility
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                '/content/rl-trading-live/requirements.txt'], check=True)
print('Dependencies installed.')

In [ ]:
# CELL 4 — CLONE OR UPDATE REPO (always gets latest code from GitHub)
import os, subprocess, sys

REPO_URL  = 'https://github.com/monty313/rl-trading-live.git'
CLONE_DIR = '/content/rl-trading-live'

if os.path.exists(CLONE_DIR):
    # Hard reset — discard any local patches Gemini may have applied
    subprocess.run(['git', '-C', CLONE_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', CLONE_DIR, 'reset', '--hard', 'origin/master'], check=True)
    print('Repo hard-reset to origin/master.')
else:
    subprocess.run(['git', 'clone', REPO_URL, CLONE_DIR], check=True)
    print('Repo cloned.')

# Always cd to repo root so relative imports and ! commands work
os.chdir(CLONE_DIR)
sys.path.insert(0, CLONE_DIR)

# Flush stale module cache so updated code is actually imported
for m in list(sys.modules):
    if m.startswith(('core', 'training', 'backtest', 'broker', 'jordan')):
        del sys.modules[m]

# Confirm we have the latest commit
r = subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True, cwd=CLONE_DIR)
print('Latest commits:\n' + r.stdout)

In [ ]:
# CELL 4b — CLEAN MANIFEST (removes stale DQN entries that no longer exist on Drive)
import json, os

MANIFEST = '/content/drive/MyDrive/RL-Trading-Checkpoints/gpu/manifest.json'
CKPT_DIR = '/content/drive/MyDrive/RL-Trading-Checkpoints/gpu'

if os.path.exists(MANIFEST):
    with open(MANIFEST) as f:
        manifest = json.load(f)

    before = len(manifest.get('checkpoints', {}))
    cleaned = {
        name: meta
        for name, meta in manifest.get('checkpoints', {}).items()
        if os.path.exists(os.path.join(CKPT_DIR, name))   # file must exist
        and meta.get('phase', 'unknown') != 'unknown'      # skip legacy DQN
    }
    manifest['checkpoints'] = cleaned
    after = len(cleaned)

    with open(MANIFEST, 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f'Manifest cleaned: {before} entries → {after} entries.')
    for name, meta in cleaned.items():
        print(f"  {name}  phase={meta['phase']}  ep={meta['episode']}  phi={meta['phi']:.4f}")
else:
    print('No manifest found — will be created fresh when training starts.')

In [ ]:
# CELL 5 — SYSTEM INSPECTION (aborts on failure)
# Output streams live — smoke_train takes ~5-10 min on first torch.compile warmup.
# You will see each check print as it completes. Do NOT interrupt unless you see ❌.
%cd /content/rl-trading-live
import subprocess, sys
r = subprocess.run([sys.executable, 'inspect_system.py'], cwd='/content/rl-trading-live')
if r.returncode != 0:
    raise RuntimeError('inspect_system.py failed — fix issues above before training')

In [ ]:
# CELL 6 — RUN TRAINING (resume from Drive checkpoints; runs indefinitely)
# NOTE: torch.compile warmup makes the first ~10-15 min / first few episodes
# slow. This is NORMAL warmup, not a crash. Let it run.
%cd /content/rl-trading-live
!python -m training.train --resume \
    --csv            "/content/drive/MyDrive/RL-Trading-Data/EURUSD_M1_202101131130_202605270000_2020_2026.csv" \
    --checkpoint-dir "/content/drive/MyDrive/RL-Trading-Checkpoints/gpu" \
    --metrics-dir    "/content/drive/MyDrive/RL-Trading-Checkpoints/metrics" \
    --manifest       "/content/drive/MyDrive/RL-Trading-Checkpoints/manifest.json"

In [ ]:
# CELL 7 — LAUNCH JORDAN DASHBOARD (localtunnel; ngrok fallback)
import subprocess, sys, time
CLONE_DIR = '/content/rl-trading-live'
subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'dashboard/app.py',
    '--server.port', '8501', '--server.headless', 'true'], cwd=CLONE_DIR)
time.sleep(6)
# localtunnel is flaky on Colab; if the URL doesn't load, use the ngrok block below.
subprocess.run(['npx', 'localtunnel', '--port', '8501'], check=False)
# Fallback (uncomment): 
# !pip -q install pyngrok && python -c "from pyngrok import ngrok; print(ngrok.connect(8501))"

In [ ]:
# CELL 8 — CRASH RECOVERY (run if training crashed)
import subprocess, sys
CLONE_DIR = '/content/rl-trading-live'
subprocess.run([sys.executable, 'scripts/crash_recovery.py',
    '--checkpoint-dir', '/content/drive/MyDrive/RL-Trading-Checkpoints/gpu',
    '--manifest', '/content/drive/MyDrive/RL-Trading-Checkpoints/manifest.json',
], cwd=CLONE_DIR, check=True)
# Then re-run CELL 6.

In [ ]:
# CELL 9 — GPU PROFILING (run after CELL 6 has trained a few episodes)
import torch, sys, os
sys.path.insert(0, '/content/rl-trading-live')
os.chdir('/content/rl-trading-live')
from torch.profiler import profile, record_function, ProfilerActivity
from core.pipeline import build_pipeline
from core.settings import CFG, get_device, auto_tune_batch

device = get_device()
cfg = auto_tune_batch(dict(CFG), device)
cfg['DATA_CSV_EURUSD'] = '/content/drive/MyDrive/RL-Trading-Data/EURUSD_M1_202101131130_202605270000_2020_2026.csv'
env, agent, *_ = build_pipeline(cfg, device,
    phase={'name': 'profile', 'entry_conditions': {'buy': 'any', 'sell': 'any'}})

obs = torch.randn(cfg['BATCH_SIZE_ENV'], env.state_dim, device=device)
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
             record_shapes=True) as prof:
    with record_function('actor_critic_forward'):
        with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
            out = agent.net(obs)   # agent.net is the ActorCritic module

print(prof.key_averages().table(sort_by='cuda_time_total', row_limit=15))
cuda_t = sum(e.cuda_time for e in prof.key_averages())
cpu_t  = sum(e.cpu_time  for e in prof.key_averages())
ratio  = cuda_t / (cuda_t + cpu_t + 1e-9)
print(f'\nGPU time ratio: {ratio:.1%}')
print('GPU utilization OK' if ratio >= 0.5 else
      'WARNING: GPU < 50% — raise BATCH_SIZE_ENV in core/settings.py')